# PFE ML Fixed Quality Pipeline

This notebook reruns the pipeline with the date-quality fixes and safer ML split rules.

What this version does:

- Re-exports BODACC raw Parquet with impossible XML dates rejected by the parser.
- Rebuilds clean tables with date bounds, so impossible dates become null and do not enter features.
- Rebuilds full uncapped feature tables in 2-year batches to reduce RAM pressure.
- Excludes company-year rows before a company creation date, preventing negative company ages.
- Runs validation checks for impossible dates and negative ages.
- Keeps 2025 as a scoring snapshot, not a supervised training/evaluation label year.


## 1. Runtime

Use **High-RAM CPU**. Do not use TPU for this pipeline. GPU is not needed for the DuckDB/Parquet build.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
WORK_DIR = '/content/pfe_work'

START_YEAR = 2017
END_YEAR = 2025
TRAIN_END_YEAR = 2024
YEAR_BATCH_SIZE = 2

DUCKDB_TMP = '/content/pfein_duckdb_tmp'
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR =', BACKEND_DIR)
print('DRIVE_ROOT  =', DRIVE_ROOT)
print('WORK_DIR    =', WORK_DIR)


In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git pull --ff-only
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt


## 2. Optional Pre-Fix Date Triage

Run this before the rebuild if you want to confirm the current lake contains impossible dates. It does not modify data.

In [ ]:
import duckdb

con = duckdb.connect()
DL = f'{DRIVE_ROOT}/data-lake'

def show(sql):
    return con.execute(sql).df()

raw_bodacc = f'{DL}/raw/bodacc/**/*.parquet'
clean_legal = f'{DL}/clean/legal_events/**/*.parquet'
features = f'{DL}/features/company_year_features/**/*.parquet'

print('Bad clean legal event years, if clean_legal exists:')
try:
    display(show(f"""
        SELECT
          COUNT(*) AS bad_rows,
          MIN(event_date) AS min_event_date,
          MAX(event_date) AS max_event_date,
          MIN(event_year) AS min_event_year,
          MAX(event_year) AS max_event_year
        FROM read_parquet('{clean_legal}', union_by_name=true)
        WHERE event_date < DATE '1900-01-01'
           OR event_date > CURRENT_DATE + INTERVAL 366 DAY
           OR event_year < 1900
           OR event_year > YEAR(CURRENT_DATE + INTERVAL 366 DAY)
    """))
except Exception as exc:
    print('clean legal check skipped:', exc)

print('Negative feature ages, if features exist:')
try:
    display(show(f"""
        SELECT prediction_year,
               COUNT(*) AS rows,
               SUM(CASE WHEN company_age_years < 0 THEN 1 ELSE 0 END) AS negative_age_rows,
               MIN(company_age_years) AS min_age,
               MAX(company_age_years) AS max_age
        FROM read_parquet('{features}', union_by_name=true)
        GROUP BY prediction_year
        ORDER BY prediction_year
    """))
except Exception as exc:
    print('feature age check skipped:', exc)


## 3. Re-Export BODACC Raw With Date Parser Fix

This step reuses already-downloaded BODACC archives from Drive. It does not redownload them. It overwrites raw BODACC Parquet so stale bad-date rows do not remain.

In [ ]:
import shlex
import subprocess
import sys

command = [
    sys.executable,
    '-u',
    'collabs/full_pipeline.py',
    '--step', 'export_raw_bodacc',
    '--force',
    '--drive-root', DRIVE_ROOT,
    '--work-dir', WORK_DIR,
    '--repo-dir', BACKEND_DIR,
    '--start-year', str(START_YEAR),
    '--end-year', str(END_YEAR),
    '--bodacc-mode', 'historical',
    '--bodacc-families', 'PCL', 'RCS-B',
    '--bodacc-overwrite-raw',
]

print(' '.join(shlex.quote(part) for part in command))
subprocess.run(command, check=True)


## 4. Rebuild Clean Tables And Full Features

This is the main fixed rebuild. It is full and uncapped. `--year-batch-size 2` only batches prediction years for memory; it does not cap companies or remove rows after the quality rules.

In [ ]:
command = [
    sys.executable,
    '-u',
    'collabs/full_pipeline.py',
    '--step', 'build_ml_data',
    '--drive-root', DRIVE_ROOT,
    '--work-dir', WORK_DIR,
    '--repo-dir', BACKEND_DIR,
    '--start-year', str(START_YEAR),
    '--end-year', str(END_YEAR),
    '--year-batch-size', str(YEAR_BATCH_SIZE),
]

print(' '.join(shlex.quote(part) for part in command))
subprocess.run(command, check=True)


## 5. Post-Fix Hard Validation

These checks should return zero impossible clean legal dates, zero negative company ages, and no feature dates outside the expected modeling window.

In [ ]:
con = duckdb.connect()
DL = f'{DRIVE_ROOT}/data-lake'
clean_legal = f'{DL}/clean/legal_events/**/*.parquet'
features = f'{DL}/features/company_year_features/**/*.parquet'
labels = f'{DL}/features/risk_labels/**/*.parquet'

bad_legal = con.execute(f"""
    SELECT COUNT(*) AS bad_rows,
           MIN(event_date) AS min_event_date,
           MAX(event_date) AS max_event_date,
           MIN(event_year) AS min_event_year,
           MAX(event_year) AS max_event_year
    FROM read_parquet('{clean_legal}', union_by_name=true)
    WHERE event_date < DATE '1900-01-01'
       OR event_date > CURRENT_DATE + INTERVAL 366 DAY
       OR event_year < 1900
       OR event_year > YEAR(CURRENT_DATE + INTERVAL 366 DAY)
""").df()
display(bad_legal)

age_check = con.execute(f"""
    SELECT prediction_year,
           COUNT(*) AS rows,
           SUM(CASE WHEN company_age_years < 0 THEN 1 ELSE 0 END) AS negative_age_rows,
           MIN(company_age_years) AS min_age,
           MAX(company_age_years) AS max_age
    FROM read_parquet('{features}', union_by_name=true)
    GROUP BY prediction_year
    ORDER BY prediction_year
""").df()
display(age_check)

label_balance = con.execute(f"""
    SELECT prediction_year,
           COUNT(*) AS rows,
           SUM(continuity_risk_12m_label::INTEGER) AS continuity_positive,
           AVG(continuity_risk_12m_label::INTEGER) AS continuity_rate
    FROM read_parquet('{labels}', union_by_name=true)
    GROUP BY prediction_year
    ORDER BY prediction_year
""").df()
display(label_balance)

if int(bad_legal['bad_rows'].iloc[0]) != 0:
    raise RuntimeError('Clean legal events still contain impossible dates.')
if int(age_check['negative_age_rows'].fillna(0).sum()) != 0:
    raise RuntimeError('Feature table still contains negative company ages.')


## 6. Audits

Run the data-lake audit first, then the ML-specific readiness audits.

In [ ]:
audit_command = [
    sys.executable,
    '-u',
    'collabs/audit_data_lake.py',
    '--drive-root', DRIVE_ROOT,
    '--max-columns', '25',
    '--sample-rows', '2',
]

print(' '.join(shlex.quote(part) for part in audit_command))
subprocess.run(audit_command, check=True)

!ls -lah "$DRIVE_ROOT/reports"
!head -120 "$DRIVE_ROOT/reports/data_lake_audit.md"


In [ ]:
!python collabs/audit_ml_readiness.py \
  --drive-root "$DRIVE_ROOT" \
  --sample-rows 20

!ls -lah "$DRIVE_ROOT/reports/label_audit.md" \
         "$DRIVE_ROOT/reports/leakage_audit.md" \
         "$DRIVE_ROOT/reports/feature_safety_audit.md" \
         "$DRIVE_ROOT/reports/ml_readiness_audit.json"

!head -100 "$DRIVE_ROOT/reports/label_audit.md"


## 7. Optional Baseline Training

Train only on labeled years through 2024. The 2025 rows are kept for scoring/current prediction because their 12-month future label window is incomplete.

Important: the current training tool loads the selected training table into pandas. A full 2017-2024 training set is too large for Colab. `TRAIN_MAX_ROWS` is a training-sample limit only; it does not cap or change the data lake.

In [ ]:
RUN_TRAINING = False
TRAIN_MAX_ROWS = 2_000_000

if RUN_TRAINING:
    train_command = [
        sys.executable,
        '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
        '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
        '--train-start-year', str(START_YEAR),
        '--train-end-year', str(TRAIN_END_YEAR),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
    ]
    print(' '.join(shlex.quote(part) for part in train_command))
    subprocess.run(train_command, check=True)
    !cat "$DRIVE_ROOT/ml-artifacts/model_metadata.json"
else:
    print('Training skipped. Set RUN_TRAINING = True after audits pass.')


## 8. What To Report

Use these points in your project notes:

- Raw public sources can contain impossible or malformed dates.
- The fixed clean layer enforces date bounds before feature generation.
- Feature generation excludes pre-creation company-year rows.
- Model training/evaluation excludes 2025 labels because the 12-month future window is incomplete.
- 2025 remains valid for current scoring after the model is trained on historical years.